# Qwen3-VL-Embedding and Qwen3-VL-Reranker: A Complete Walkthrough

A hands-on tour of Alibaba's January 2026 multimodal retrieval suite. Both models are 2B parameters here, so the full notebook runs on a free Colab T4 GPU.

## What you will learn

- What each of these models actually does and how they differ
- The full input / output JSON schema for both models
- Text-to-image, image-to-image, screenshot / document, mixed-modal, and short-video retrieval
- The native video API (frame sampling at a chosen fps)
- Instruction-aware retrieval, multilingual queries, Matryoshka embedding dimensions, pure text-to-text similarity
- The two-stage Embedding → Reranker pipeline

## How to run

- Free Colab T4 (16 GB VRAM) is enough. We use the 2B variants throughout.
- Click `Runtime` then `Run all`. The first run downloads roughly 7 GB of weights, so budget about five minutes.
- A few cells near the end accept your own queries.

## Reference

- Paper: [Qwen3-VL-Embedding and Qwen3-VL-Reranker, arXiv:2601.04720](https://arxiv.org/abs/2601.04720)
- Models: [Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B), [Reranker-2B](https://huggingface.co/Qwen/Qwen3-VL-Reranker-2B)
- Code repo: [QwenLM/Qwen3-VL-Embedding](https://github.com/QwenLM/Qwen3-VL-Embedding)

## 1. Setup

We install the required libraries, clone the official Qwen3-VL-Embedding repo (both `Qwen3VLEmbedder` and `Qwen3VLReranker` live there, under `src/models/`), and load the two 2B models on the T4.

Two T4-specific notes worth flagging up front:

1. T4 is a Turing-class GPU, so we use `torch.float16`. `bfloat16` is Ampere+ only and will silently degrade or error.
2. Flash-Attention 2 also requires Ampere+, so we leave attention at PyTorch's default SDPA.

In [ ]:
# Install dependencies. The quiet flag keeps the output short.
!pip install -q "transformers>=4.57.0" "qwen-vl-utils>=0.0.14" \
                datasets pillow matplotlib accelerate \
                opencv-python-headless scipy

In [ ]:
# Clone the official Qwen3-VL-Embedding repo for both model classes.
# Both Qwen3VLEmbedder and Qwen3VLReranker live in src/models/ inside this repo.
# (os and sys are imported here because this cell runs BEFORE the main imports cell.)
import os, sys
if not os.path.exists("Qwen3-VL-Embedding"):
    !git clone --depth 1 https://github.com/QwenLM/Qwen3-VL-Embedding.git
sys.path.insert(0, os.path.abspath("Qwen3-VL-Embedding"))

In [ ]:
ASSETS_BASE = "https://raw.githubusercontent.com/Sudip-329/learnopencv/master/Qwen3-VL-Embedding-Reranker/assets"

In [ ]:
# Core imports
import os, sys, io, json, time, gc, subprocess, textwrap
import requests
import torch
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from pathlib import Path
from PIL import Image
from IPython.display import display, clear_output

# Embedding and Reranker classes from the cloned repo
from src.models.qwen3_vl_embedding import Qwen3VLEmbedder
from src.models.qwen3_vl_reranker import Qwen3VLReranker

# GPU and dtype check
device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if device == "cuda" else torch.float32  # T4 = Turing -> fp16
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"dtype:  {DTYPE}")


In [ ]:
# Load Qwen3-VL-Embedding-2B (the bi-encoder)
print("Loading Qwen3-VL-Embedding-2B ...")
t0 = time.time()
embedder = Qwen3VLEmbedder(
    model_name_or_path="Qwen/Qwen3-VL-Embedding-2B",
    torch_dtype=DTYPE,
)
print(f"Loaded in {time.time() - t0:.1f} s")

In [ ]:
# Load Qwen3-VL-Reranker-2B (the cross-encoder)
print("Loading Qwen3-VL-Reranker-2B ...")
t0 = time.time()
reranker = Qwen3VLReranker(
    model_name_or_path="Qwen/Qwen3-VL-Reranker-2B",
    torch_dtype=DTYPE,
)
print(f"Loaded in {time.time() - t0:.1f} s")

In [ ]:
# Tiny helpers we'll use throughout
def encode(items, batch_size=4):
    """Encode a list of input dicts to a (N, 2048) tensor.

    Each item is a dict like {'text': ...}, {'image': url_or_path}, or
    {'text': ..., 'image': ...}. We batch in small groups so 20 items
    don't OOM the T4 in a single forward pass. Embeddings come back
    L2-normalised (the `process` method does this by default).
    """
    if not items:
        raise ValueError("encode() called with an empty list. "
                         "Check that the image corpus loaded correctly.")
    chunks = []
    for i in range(0, len(items), batch_size):
        chunks.append(embedder.process(items[i:i + batch_size]))
    return torch.cat(chunks, dim=0)

def cosine_topk(query_emb, doc_embs, k=3):
    """query_emb: (1, D), doc_embs: (N, D), both L2-normalised.
    Returns (top_k_indices, top_k_scores, full_sims).
    Cosine == dot product when both sides are unit-norm."""
    sims = (query_emb @ doc_embs.T)[0]  # shape (N,)
    k = min(k, sims.shape[0])
    idxs = sims.argsort(descending=True)[:k].tolist()
    scores = [sims[i].item() for i in idxs]
    return idxs, scores, sims

def as_json_result(query, idxs, scores, captions=None, urls=None):
    """Return a structured dict describing a retrieval result so we
    can print it with json.dumps(..., indent=2)."""
    items = []
    for rank, (i, s) in enumerate(zip(idxs, scores), start=1):
        item = {"rank": rank, "index": int(i), "score": round(float(s), 4)}
        if captions is not None and i < len(captions): item["caption"] = captions[i]
        if urls     is not None and i < len(urls):     item["url"]     = urls[i]
        items.append(item)
    return {"query": query, "top_k": items}

def to_square(img, size=400, bg=(240, 240, 240)):
    """Letterbox a PIL image to a fixed square so all displayed images are uniform.
    Resizes preserving aspect ratio, then pastes centered on a `size`x`size` canvas."""
    img = img.copy().convert("RGB")
    img.thumbnail((size, size), Image.LANCZOS)
    canvas = Image.new("RGB", (size, size), bg)
    canvas.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
    return canvas

def middle_frame(video_path):
    """Read the middle frame of a video file and return it as a PIL Image (RGB)."""
    cap = cv2.VideoCapture(video_path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, n // 2)
    ok, frame = cap.read()
    cap.release()
    return Image.fromarray(frame[:, :, ::-1]) if ok else None

def cache_video(url_or_path, cache_dir="video_cache"):
    """Return a local file path for the given URL or path. If it's a URL,
    download to `cache_dir` on first call and cache thereafter."""
    if not url_or_path.startswith(("http://", "https://")):
        return url_or_path
    os.makedirs(cache_dir, exist_ok=True)
    local_path = os.path.join(cache_dir, os.path.basename(url_or_path))
    if not os.path.exists(local_path) or os.path.getsize(local_path) == 0:
        r = requests.get(url_or_path, timeout=120, verify=False)
        r.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(r.content)
    return local_path


### Sanity check

A one-second smoke test before the heavy work. We encode two paraphrase sentences plus one unrelated mountain sentence, then check the 3×3 similarity matrix. A healthy load shows 1.0 on the diagonal (mathematically guaranteed for unit vectors), paraphrase similarity in roughly the 0.6-0.7 range, and unrelated similarity in roughly the 0.2-0.3 range. The *gap* between them is what matters. If it collapses, the model load is broken and every downstream demo will silently produce garbage.

In [ ]:
sanity_inputs = [
    {"text": "a black cat sitting on a chair"},
    {"text": "a sleek dark feline resting on furniture"},
    {"text": "an aerial photograph of a snowy mountain range"},
]
print("Inputs (JSON):")
print(json.dumps(sanity_inputs, indent=2))

sanity = encode(sanity_inputs)
print(f"\nEmbedding shape: {tuple(sanity.shape)}")
sim_matrix = (sanity @ sanity.T).round(decimals=3)
print("\nSimilarity matrix (rows/cols match the three strings above):")
print(sim_matrix)

## 2. What each model actually does

These two models are **not** alternatives. They are designed to be used together in a retrieval pipeline. Confusing them is the single most common mistake when first reading the model cards.

| | Qwen3-VL-Embedding-2B | Qwen3-VL-Reranker-2B |
|---|---|---|
| **Architecture** | Bi-encoder (dual-tower) | Cross-encoder (single-tower) |
| **Input** | One item at a time (text, image, screenshot, video, or a mix) | A `(query, document)` pair, each of any modality |
| **Output** | A 2048-dim L2-normalised vector | A single relevance score in `[0, 1]` |
| **Best at** | Indexing a corpus once, then doing similarity search forever | Looking very carefully at a small set of candidates |
| **Cost per query** | One vector compare per item (fast, vectorizable) | One forward pass per pair (slow) |
| **Role in a search system** | Stage 1 (recall): top-K from millions | Stage 2 (rerank): top-K to top-3 |

> **A note on the reranker output range.** The `[0, 1]` claim assumes you call `Qwen3VLReranker.process(...)` from the official repo, which applies a sigmoid internally. The sentence-transformers path (`CrossEncoder.predict(pairs)`) returns *unbounded* raw logits by default; pass `activation_fn=torch.nn.Sigmoid()` to get the same `[0, 1]` behavior. The notebook uses `.process(...)`, so all scores you see are in `[0, 1]`.

## 2b. Input and output format reference

Before we run any demos, here is the full schema for what you can feed each model and what comes back. Both models use the same `.process(inputs)` method, but their input shape and output shape are different.

### Embedding: `Qwen3VLEmbedder.process(inputs)`

**Input.** A `list[dict]`. Each dict represents one item to embed and may contain any combination of these keys:

```json
[
  { "text": "a search query or document text" },
  { "text": "...", "instruction": "Retrieve relevant images." },
  { "image": "https://example.com/image.jpg" },
  { "image": "/path/to/local/image.jpg" },
  { "image": "<PIL.Image instance>" },
  { "text": "shoes like these but in black", "image": "https://example.com/shoe.jpg" },
  { "video": "/path/to/video.mp4", "fps": 1.0, "max_frames": 64 }
]
```

**Output.** A `torch.Tensor` of shape `(len(inputs), 2048)`, dtype matches `torch_dtype` (fp16 here), L2-normalised. Cosine similarity therefore equals dot product.

### Reranker: `Qwen3VLReranker.process(inputs)`

**Input.** A single `dict` (not a list):

```json
{
  "instruction": "Retrieve images or text relevant to the user's query.",
  "query":     { "text": "a query string" },
  "documents": [
    { "text": "candidate document 1" },
    { "image": "https://example.com/candidate2.jpg" },
    { "text": "candidate 3 caption", "image": "https://example.com/candidate3.jpg" }
  ],
  "fps":        1.0,
  "max_frames": 64
}
```

Both `query` and each entry in `documents` can be a text-only, image-only, video-only, or mixed dict using the same keys as the embedder.

**Output.** A `list[float]` with one relevance score in `[0, 1]` per document, in the same order as the input.

### What is *not* supported

- Plain strings instead of dicts. You must wrap text as `{"text": "..."}`.
- Bytes / numpy arrays for images. Use a PIL Image, a path, or a URL.
- Multiple documents in the embedder's input. Each list element is one independent item; use a list comprehension to encode a corpus.

## 3. Text-to-image retrieval

The classic multimodal search task: given a text query, find the most relevant images in a corpus.

In [ ]:
OPENCV = "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/"

CORPUS = [
    # Qwen's own demo image (always reachable, different CDN)
    ("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg", "woman with golden retriever on beach"),
    # OpenCV sample images
    (OPENCV + "aero1.jpg",            "aerial view of a city from above"),
    (OPENCV + "aero3.jpg",            "aerial view of a coastal town from above"),
    (OPENCV + "baboon.jpg",           "baboon face close-up"),
    (OPENCV + "butterfly.jpg",        "butterfly resting on a green striped leaf"),
    (OPENCV + "fruits.jpg",           "assortment of fresh fruits"),
    (OPENCV + "apple.jpg",            "single red apple"),
    (OPENCV + "orange.jpg",           "single orange fruit"),
    (OPENCV + "starry_night.jpg",     "van Gogh starry night painting"),
    (OPENCV + "messi5.jpg",           "soccer player on the field"),
    (OPENCV + "sudoku.png",           "sudoku puzzle grid"),
    (OPENCV + "chessboard.png",       "chessboard calibration pattern"),
    (OPENCV + "building.jpg",         "tall building exterior"),
    (OPENCV + "home.jpg",             "ornate yellow palace with domes and birds in the sky"),
    (OPENCV + "graf1.png",            "outdoor graffiti wall"),
    (OPENCV + "stuff.jpg",            "desk with miscellaneous objects"),
    (OPENCV + "HappyFish.jpg",        "cartoon fish illustration"),
    (OPENCV + "Blender_Suzanne1.jpg", "close-up of a flat-shaded grey 3D polygon model"),
    (OPENCV + "box_in_scene.png",     "grayscale photo of grocery boxes on a chair"),
    (OPENCV + "smarties.png",         "scattered colorful candies"),
    # ---- New high-quality additions (loaded from Drive in dev / GitHub raw at publish) ----
    (f"{ASSETS_BASE}/qwen3vl_cat_portrait.jpg",  "tabby cat studio portrait"),
    (f"{ASSETS_BASE}/qwen3vl_dog_portrait.jpg",  "Border Collie standing in a city park"),
    (f"{ASSETS_BASE}/qwen3vl_dog_on_beach.jpg",  "yellow Labrador running on a beach at sunset"),
    (f"{ASSETS_BASE}/qwen3vl_beach_empty.jpg",   "empty sunlit beach with golden sand and waves"),
    (f"{ASSETS_BASE}/qwen3vl_dog_in_snow.jpg",   "yellow Labrador standing in snowy mountains"),
    (f"{ASSETS_BASE}/qwen3vl_meal_plated.jpg",   "plated salmon dinner with vegetables and lemon"),
]

# Disable SSL warnings (some CDNs require verify=False on Colab).
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def load_image(src, timeout=20):
    """Load an image from a URL or a local file path. Returns a PIL Image (RGB)."""
    if src.startswith(("http://", "https://")):
        r = requests.get(src, timeout=timeout, verify=False)
        r.raise_for_status()
        if not r.content:
            raise IOError(f"empty body returned for {src}")
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    # Local file path
    if not os.path.exists(src):
        raise FileNotFoundError(f"local asset missing: {src}")
    return Image.open(src).convert("RGB")

images, captions, urls = [], [], []
print(f"Loading {len(CORPUS)} images (HTTP for URLs, direct read for local paths) ...")
for src, cap in CORPUS:
    try:
        images.append(load_image(src))
        captions.append(cap)
        urls.append(src)
    except Exception as e:
        print(f"  skipped: {cap} ({type(e).__name__}: {e})")
print(f"\nLoaded {len(images)} images.")

assert len(images) >= 5, (
    f"Only {len(images)} images downloaded — corpus too small. "
    "Check your internet connection and the URLs above."
)

In [ ]:
# Visualise the corpus
def show_grid(imgs, titles=None, cols=5, row_h=3.0, title_fontsize=8):
    if not imgs:
        print("(no images to display)")
        return
    cols = max(1, min(cols, len(imgs)))
    rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, row_h * rows))
    axes = [axes] if rows * cols == 1 else axes.flatten()
    for i, ax in enumerate(axes):
        if i < len(imgs):
            ax.imshow(imgs[i])
            if titles is not None:
                ax.set_title(titles[i], fontsize=title_fontsize)
        ax.axis("off")
    plt.tight_layout(); plt.show()

show_grid(images, captions, cols=5)

In [ ]:
# Encode the entire image corpus once. Inputs follow the schema from section 2b.
print("Sample input (JSON):")
print(json.dumps([{"image": urls[0]}, {"image": urls[1]}], indent=2))

print(f"\nEncoding {len(images)} images ...")
t0 = time.time()
image_inputs = [{"image": u} for u in urls]
image_embeddings = encode(image_inputs, batch_size=4)
print(f"Done in {time.time() - t0:.1f} s -> shape {tuple(image_embeddings.shape)}")

In [ ]:
# Run a few text queries and visualise the top-3 retrievals for each.
TEXT_QUERIES = [
    "a pair of fruit items",
    "an aerial view of a city",
    "a wild animal photograph",
    "a famous painting",
    "a logical puzzle from a newspaper",
]

def topk_text_to_image(query, k=3):
    q_emb = encode([{"text": query}])
    idxs, scores, _ = cosine_topk(q_emb, image_embeddings, k=k)
    return idxs, scores

for query in TEXT_QUERIES:
    idxs, scores = topk_text_to_image(query, k=3)

    # JSON-style structured result
    result = as_json_result(query, idxs, scores, captions=captions, urls=urls)
    print(json.dumps(result, indent=2))

    # Visual: tight wspace, room reserved for the Query suptitle
    fig, axes = plt.subplots(
        1, 3,
        figsize=(12, 4.2),
        gridspec_kw={"wspace": 0.05},
    )
    axes = [axes] if not hasattr(axes, "__len__") else axes
    for ax, i, s in zip(axes, idxs, scores):
        ax.imshow(images[i])
        ax.set_title(f"sim={s:.3f}\n{captions[i][:45]}", fontsize=9)
        ax.axis("off")
    fig.suptitle(f'Query: "{query}"', fontsize=12)
    plt.subplots_adjust(top=0.82, wspace=0.05)   # 18% reserved for suptitle
    plt.show()
    print("-" * 60)

### 🔍 What does `sim=###` mean?
A similarity score between your text query and each retrieved image. **Higher means more relevant** — the top-ranked image is the model's best guess at what you asked for. Scores are roughly between 0 and 1, and for text-to-image search they typically land in the 0.2–0.4 range. What really matters is the *ranking*, not the absolute number: whichever image has the highest score is the model's #1 pick.

**What just happened.** The Embedding model gave every image a 2048-dim L2-normalised vector. The text query gets a vector from the same encoder. The model is one model with shared weights for text and image inputs, but it does not see them together in this mode. A simple dot product (which equals cosine similarity for unit vectors) ranks the corpus.

The JSON output above is what you would store in a search backend: each retrieval is a structured record you can ship to a frontend, log to a database, or feed into an evaluation harness.

## 4. Image-to-image retrieval

The same Embedding model does "find visually similar images" with no code change. We already have a vector for every corpus image; we just compare against another image vector instead of a text vector.

In [ ]:
# Pick one image as the query, find its nearest neighbours in the same corpus.
QUERY_IDX = 0  # the Qwen demo image (woman with golden retriever on beach)

q_input = {"image": urls[QUERY_IDX]}
print("Query input (JSON):")
print(json.dumps(q_input, indent=2))

q_emb = encode([q_input])
sims = (q_emb @ image_embeddings.T)[0]
# Sort and exclude the query itself
ranked = sorted(enumerate(sims.tolist()), key=lambda x: -x[1])
neighbours = [(i, s) for i, s in ranked if i != QUERY_IDX][:4]

# Structured output
print("\nResult (JSON):")
print(json.dumps({
    "query_image_index": QUERY_IDX,
    "query_caption":     captions[QUERY_IDX],
    "nearest_neighbours": [
        {"rank": r+1, "index": i, "score": round(s, 4), "caption": captions[i]}
        for r, (i, s) in enumerate(neighbours)
    ],
}, indent=2))

fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
axes[0].imshow(images[QUERY_IDX])
axes[0].set_title("QUERY", fontsize=10, weight="bold"); axes[0].axis("off")
for ax, (i, s) in zip(axes[1:], neighbours):
    ax.imshow(images[i])
    ax.set_title(f"sim={s:.3f}\n{captions[i][:25]}", fontsize=9)
    ax.axis("off")
plt.suptitle("Image-to-image retrieval", fontsize=11)
plt.tight_layout(); plt.show()

**A note on what gets retrieved.** Without an explicit instruction, the model captures a mix of semantic similarity ("contains a dog", "outdoor scene") and visual similarity ("warm tones", "people in the frame"). For pure visual similarity, you can pass an instruction inside the input dict, for example `{"image": url, "instruction": "Find visually similar images."}`. We will see instructions in action in section 7c.

## 5. Logo, screenshot and document retrieval

Qwen3-VL was trained on a lot of screen and document data, so its embeddings are unusually good at finding *the right logo*, *the right interface*, or *the right page* by a text query. This is useful for support knowledge bases, internal documentation, brand-asset search, or building a "search my screenshots" tool.

The OpenCV samples folder ships a few real brand logos and screenshot-like images. We mix them with photo distractors and search.

In [ ]:
# A small corpus of logos and screenshot-like images plus photo distractors.
# All from the OpenCV samples folder, so they're reliable on Colab.
DOC_CORPUS = [
    (OPENCV + "LinuxLogo.jpg",         "Linux operating system logo (Tux penguin)"),
    (OPENCV + "WindowsLogo.jpg",       "Windows operating system logo"),
    (OPENCV + "opencv-logo.png",       "OpenCV library logo (transparent)"),
    (OPENCV + "opencv-logo-white.png", "OpenCV library logo (white)"),
    (OPENCV + "sudoku.png",            "sudoku puzzle screenshot"),
    (OPENCV + "chessboard.png",        "chessboard calibration screenshot"),
    # Photo distractors that should NOT match logo / UI queries
    (OPENCV + "baboon.jpg",            "baboon photograph (distractor)"),
    (OPENCV + "butterfly.jpg",         "butterfly photograph (distractor)"),
    (OPENCV + "fruits.jpg",            "fruits photograph (distractor)"),
]

doc_images, doc_captions, doc_urls = [], [], []
for url, cap in DOC_CORPUS:
    try:
        doc_images.append(load_image(url))
        doc_captions.append(cap)
        doc_urls.append(url)
    except Exception as e:
        print(f"  skipped {cap}: {type(e).__name__}")

print(f"Loaded {len(doc_images)} items.")
show_grid(doc_images, doc_captions, cols=min(5, len(doc_images)), row_h=3.0)

In [ ]:
# Encode the doc corpus and run queries that target logo / screenshot content
doc_embs = encode([{"image": u} for u in doc_urls], batch_size=3)

UI_QUERIES = [
    "an operating system mascot or logo",
    "a software library brand",
    "a logic puzzle screenshot",
    "a calibration pattern image",
    "a photograph of an animal",
]
for query in UI_QUERIES:
    q = encode([{"text": query}])
    idxs, scores, _ = cosine_topk(q, doc_embs, k=3)
    result = as_json_result(query, idxs, scores, captions=doc_captions, urls=doc_urls)
    print(json.dumps(result, indent=2))
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    for ax, i, s in zip(axes, idxs, scores):
        ax.imshow(doc_images[i])
        ax.set_title(f"{s:.3f} | {doc_captions[i][:30]}", fontsize=9)
        ax.axis("off")
    fig.suptitle(f'Query: "{query}"', fontsize=11)
    plt.tight_layout(); plt.show()
    print("-" * 60)

The animal-photo distractors should rank below the logos for all UI-flavoured queries, and the baboon should jump to the top for the photo query. The model distinguishes brand and UI content from photos by semantic content, not just texture statistics. The same pattern scales to your own screenshot folder.

### Threshold-based retrieval: letting the model say "no match"

Instead of forcing the model to return exactly 3 results, we apply a **confidence threshold** on the similarity scores — the model returns however many items pass the cutoff, which could be 0, 1, or many. Watch the *"wild animal"* query: with no real animal photo in the corpus, the model can now honestly answer "no match" instead of inventing one.

In [ ]:

THRESHOLD = 0.30   # confidence cutoff — see the guide below to tune

UI_QUERIES = [
    "the Linux operating system mascot",
    "the Windows brand logo",
    "the OpenCV computer vision library logo",
    "a photograph of a wild animal",
    "a logic puzzle screenshot",
]

# 1) Print the structured result per query
for query in UI_QUERIES:
    q_emb     = encode([{"text": query}])
    sims      = (q_emb @ doc_embs.T)[0].tolist()
    ranked    = sorted(enumerate(sims), key=lambda x: -x[1])
    confident = [(i, s) for i, s in ranked if s >= THRESHOLD]

    result = {
        "query":     query,
        "threshold": THRESHOLD,
        "n_matches": len(confident),
        "matches": [
            {"score": round(s, 3), "caption": doc_captions[i]}
            for i, s in confident
        ],
        "verdict": (
            "no confident match" if not confident else
            f"{len(confident)} confident match(es)"
        ),
        "best_below_threshold": (
            None if len(confident) == len(ranked) else
            {"score":   round(ranked[len(confident)][1], 3),
             "caption": doc_captions[ranked[len(confident)][0]]}
        ),
    }
    print(json.dumps(result, indent=2))
    print('-' * 60)

In [ ]:
# 2) Visual: bar chart per query of ALL scores, with the threshold line
n_q = len(UI_QUERIES)
fig, axes = plt.subplots(n_q, 1, figsize=(10, 2.5 * n_q))
if n_q == 1:
    axes = [axes]

for ax, query in zip(axes, UI_QUERIES):
    q_emb  = encode([{"text": query}])
    sims   = (q_emb @ doc_embs.T)[0].tolist()
    ranked = sorted(zip(range(len(sims)), sims), key=lambda x: -x[1])

    labels = [doc_captions[i][:35] for i, _ in ranked]
    scores = [s for _, s in ranked]
    colors = ["#2ca02c" if s >= THRESHOLD else "#cccccc" for s in scores]

    ax.barh(range(len(scores)), scores, color=colors)
    ax.set_yticks(range(len(scores)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.axvline(THRESHOLD, color="red", linestyle="--", linewidth=1.2)
    ax.text(THRESHOLD + 0.005, len(scores) - 0.5,
            f"threshold = {THRESHOLD}", color="red", fontsize=8, va="bottom")
    ax.set_xlim(0, max(0.6, max(scores) * 1.1))
    ax.set_xlabel("similarity score")
    ax.set_title(f'Query: "{query}"', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Mixed-modal queries (image + text together)

A standout capability of Qwen3-VL-Embedding is that a single input can be *both* an image and a text caption, producing one embedding for that combined input. The text and image are *blended* into one vector, so the result is sensitive to both signals at once.

To make the effect crisp, we set up a **paired-variant triplet** in the corpus:

- `qwen3vl_dog_on_beach.jpg` — anchor: a yellow Labrador running on a sunset beach
- `qwen3vl_beach_empty.jpg` — same beach, no dog (variant for "remove the subject")
- `qwen3vl_dog_in_snow.jpg` — same yellow Labrador, snowy mountains (variant for "change the scene")

We will issue two mixed-modal queries: one that should pull the "remove subject" variant up, and one that should pull the "change scene" variant up. If the model is genuinely blending image and text, the targeted variant should outrank what either text-alone or image-alone retrieval would surface.

In [ ]:
# Locate anchor and expected target in the corpus
def idx_of(needle):
    for i, u in enumerate(urls):
        if needle in u:
            return i
    raise ValueError(f"Not found in corpus: {needle}")

ANCHOR_IDX = idx_of("qwen3vl_dog_on_beach")
SNOW_IDX   = idx_of("qwen3vl_dog_in_snow")
print(f"anchor (dog on beach):          index {ANCHOR_IDX}")
print(f"expected top match (dog in snow): index {SNOW_IDX}")

In [ ]:
# A mixed-modal query: image (the anchor) + text steering to a different scene.
queries = [
    {
        "label": "change scene",
        "input": {
            "image": urls[ANCHOR_IDX],
            "text":  "the same yellow Labrador dog but standing in snowy mountains",
        },
        "expected_top_caption_keyword": "snow",
    },
]

print("Mixed-modal input (JSON):")
print(json.dumps([q["input"] for q in queries], indent=2))

In [ ]:
# Encode and retrieve top-5 for each mixed-modal query
results = []
for q in queries:
    qe   = encode([q["input"]])
    sims = (qe @ image_embeddings.T)[0]
    top  = sims.argsort(descending=True)[:5].tolist()
    results.append({
        "label":  q["label"],
        "input":  q["input"],
        "top_5": [
            {"rank": r+1, "score": round(sims[i].item(), 4), "caption": captions[i]}
            for r, i in enumerate(top)
        ],
    })

print(json.dumps(results, indent=2))

In [ ]:
q    = queries[0]
qe   = encode([q["input"]])
sims = (qe @ image_embeddings.T)[0]
top  = sims.argsort(descending=True)[:3].tolist()

fig, axes = plt.subplots(1, 4, figsize=(16, 4),
                         gridspec_kw={"wspace": 0.11})

# Col 0: anchor + query text (word-wrapped over multiple lines)
axes[0].imshow(to_square(images[ANCHOR_IDX]))
wrapped = "\n".join(textwrap.wrap(q["input"]["text"], width=30))
axes[0].set_title(f"ANCHOR + text:\n{wrapped}", fontsize=8, weight="bold")
axes[0].axis("off")

# Cols 1..3: top-3 results (caption shortened with ellipsis if too long)
for col, i in enumerate(top, start=1):
    axes[col].imshow(to_square(images[i]))
    short_cap = textwrap.shorten(captions[i], width=30, placeholder="…")
    axes[col].set_title(f"#{col}  sim={sims[i].item():.3f}\n{short_cap}",
                        fontsize=8)
    axes[col].axis("off")

plt.suptitle("Mixed-modal: anchor image + text steers the result", fontsize=12)
plt.subplots_adjust(top=0.82, wspace=0.18)
plt.show()

**What to look for.** The result should rank `qwen3vl_dog_in_snow.jpg` highest (often top-1, sometimes #2 just behind the anchor image itself) because the text *"snowy mountains"* re-steers the model away from the beach scene the image alone would imply, while the image anchors the breed and pose.

A pure text query of *"yellow Labrador in snow"* would not know exactly which dog you mean. A pure image query of the dog-on-beach picture would just retrieve other beach scenes. Only the mixed input is precise enough to single out the *same dog* in a *different setting*.

The principle scales: for product search you can give an image of a chair and the text *"in walnut wood"* and get walnut chairs without ever writing the full structural description by hand.

## 7. Short video clip retrieval (middle-frame trick)

Video inputs are supported by both models. There are two practical ways to handle them:

1. **Middle-frame approximation** (this section): pull one frame from each window of the clip and treat each as an image. Fast, simple, throws away temporal info — perfect for showing "what happens at different points in this clip" with cheap image-style encoding.
2. **Full video API** (section 7b): pass `{"video": path, "fps": 1.0}`. The model samples frames internally and produces a single embedding for the entire clip. Heavier on memory and time, but captures temporal content.

For this section we sample 6 evenly-spaced frames from the traffic video (`qwen3vl_video_traffic.mp4`, 20 s, fixed camera looking at a city intersection) and display them as a 3 × 2 grid. Different windows in the clip capture different moments of traffic — a query like "cars driving through an intersection" versus "pedestrians crossing the street" should rank different frames first.

In [ ]:
VIDEO_PATH = cache_video(f"{ASSETS_BASE}/qwen3vl_video_traffic.mp4")
size_mb = os.path.getsize(VIDEO_PATH) / 1e6
print(f"Using {VIDEO_PATH} ({size_mb:.2f} MB)")

In [ ]:
# Read 6 frames from evenly-spaced windows in the video, save as JPEGs,
# and store the paths and PIL images for the rest of section 7.
cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_native   = cap.get(cv2.CAP_PROP_FPS) or 25.0
print(f"Video: {total_frames} frames @ {fps_native:.1f} fps "
      f"(~{total_frames / fps_native:.1f} s)")

NUM_CLIPS = 6
frame_indices = [int(total_frames * (k + 0.5) / NUM_CLIPS) for k in range(NUM_CLIPS)]

os.makedirs("clips", exist_ok=True)
frame_paths = []
for k, fi in enumerate(frame_indices):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError(f"Could not read frame {fi}")
    out_path = f"clips/clip_{k}.jpg"
    Image.fromarray(frame[:, :, ::-1]).save(out_path)   # BGR -> RGB
    frame_paths.append(out_path)
cap.release()

frame_imgs = [Image.open(p) for p in frame_paths]

# Display the 6 frames as a tight 3 x 2 grid
fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 6),
    gridspec_kw={"wspace": 0.02, "hspace": 0.18},
)
for i, ax in enumerate(axes.flatten()):
    if i < len(frame_imgs):
        ax.imshow(frame_imgs[i])
        ax.set_title(f"clip_{i}", fontsize=10)
    ax.axis("off")
plt.show()


In [ ]:
# Encode the frames and run a couple of text queries
frame_embs = encode([{"image": p} for p in frame_paths], batch_size=3)

VIDEO_QUERIES = [
    "a bus crossing the intersection",
    "multiple cars at a crosswalk",
    "an empty street with no vehicles",
]

for q in VIDEO_QUERIES:
    qemb    = encode([{"text": q}])
    sims    = (qemb @ frame_embs.T)[0].tolist()
    ranking = sorted(enumerate(sims), key=lambda x: -x[1])

    # JSON output
    print(json.dumps({
        "query": q,
        "ranking": [{"clip": f"clip_{i}", "score": round(s, 4)} for i, s in ranking],
    }, indent=2))

    # Visualization: 3 image panels + 1 bar-chart panel of all 6 scores
    fig, axes = plt.subplots(
        1, 4,
        figsize=(18, 4.5),
        gridspec_kw={"wspace": 0.25, "width_ratios": [1, 1, 1, 1.3]},
    )

    # Top-3 frames
    for col, (idx, score) in enumerate(ranking[:3]):
        axes[col].imshow(frame_imgs[idx])
        axes[col].set_title(f"#{col+1}  clip_{idx}\nscore={score:.3f}", fontsize=10)
        axes[col].axis("off")

    # All-6 scores as a ranked horizontal bar chart
    clip_labels = [f"clip_{i}" for i, _ in ranking]
    scores_only = [s for _, s in ranking]
    bar_colors  = ["#2ca02c" if r < 3 else "#cccccc" for r in range(len(ranking))]
    axes[3].barh(range(len(scores_only)), scores_only, color=bar_colors)
    axes[3].set_yticks(range(len(scores_only)))
    axes[3].set_yticklabels(clip_labels, fontsize=9)
    axes[3].invert_yaxis()
    axes[3].set_xlabel("similarity score", fontsize=9)
    axes[3].set_title("all 6 scores (ranked)", fontsize=10)
    axes[3].set_xlim(0, max(scores_only) * 1.15)

    plt.suptitle(f'Query: "{q}"', fontsize=12)
    plt.subplots_adjust(top=0.82, wspace=0.25)
    plt.show()
    print("-" * 60)

## 7b. Full video API: video-to-video retrieval across a clip corpus

The middle-frame approach throws away temporal information. The native video API samples frames internally with the model and produces a single embedding for the *whole clip*. This unlocks the most useful pattern in practice: **video retrieval against a corpus of clips** — "of these four clips, which one matches my text query best?"

We use all four clips in `assets/` as a tiny video corpus:

- `qwen3vl_video_cooking.mp4` — a chef preparing food in a studio (20 s, fixed camera)
- `qwen3vl_video_traffic.mp4` — busy city street intersection (20 s, fixed camera)
- `qwen3vl_video_forest.mp4` — misty pine forest at dawn (10 s, static)
- `qwen3vl_video_waterfall.mp4` — tropical waterfall in jungle (10 s, static)

Each clip is encoded once with the full video API, then we run text queries against the resulting `(4, 2048)` matrix exactly like image retrieval — but the corpus items are *clips*, not frames.

In [ ]:
# Video corpus: (path, caption). Captions are for display only.
VIDEO_CORPUS = [
    (f"{ASSETS_BASE}/qwen3vl_video_cooking.mp4",   "a chef preparing food in a studio"),
    (f"{ASSETS_BASE}/qwen3vl_video_traffic.mp4",   "city street traffic at an intersection"),
    (f"{ASSETS_BASE}/qwen3vl_video_forest.mp4",    "misty pine forest at dawn"),
    (f"{ASSETS_BASE}/qwen3vl_video_waterfall.mp4", "tropical waterfall in a jungle"),
]
video_paths    = [p for p, _ in VIDEO_CORPUS]
video_captions = [c for _, c in VIDEO_CORPUS]

# Native video inputs. The model samples `fps` frames per second up to `max_frames`.
# On a T4 we keep both modest so the demo finishes quickly.
video_inputs = [
    {"video": p, "fps": 1.0, "max_frames": 16} for p in video_paths
]
print("Video corpus inputs (JSON):")
print(json.dumps(video_inputs, indent=2))

In [ ]:
# Encode the full video corpus once. Each clip becomes ONE 2048-dim vector.
print(f"Encoding {len(video_inputs)} videos with the native video API ...")
t0 = time.time()
video_embs = embedder.process(video_inputs)
print(f"Done in {time.time() - t0:.1f} s -> shape {tuple(video_embs.shape)}")

In [ ]:
# 1) Free up VRAM before encoding (Colab accumulates cached tensors over time).
gc.collect()
torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f"Free VRAM before encoding: {free_gb:.2f} GB")

video_inputs = [
    {"video": cache_video(p), "fps": 1.0, "max_frames": 4}
    for p in video_paths
]
print("Video corpus inputs (max_frames=4):")
print(json.dumps(video_inputs, indent=2))

# 3) Encode one at a time and clear cache between each video.
print(f"\nEncoding {len(video_inputs)} videos with the native video API ...")
t0 = time.time()
video_embs_list = []
for i, vi in enumerate(video_inputs):
    print(f"  [{i+1}/{len(video_inputs)}] {os.path.basename(vi['video'])} ...")
    emb = embedder.process([vi])
    video_embs_list.append(emb)
    torch.cuda.empty_cache()                # release fragments between videos
video_embs = torch.cat(video_embs_list, dim=0)
print(f"Done in {time.time() - t0:.1f} s -> shape {tuple(video_embs.shape)}")

# 4) Sanity check: all 4 vectors must DIFFER (off-diagonal should not be 1.0).
print("\nSelf-similarity matrix (diagonal=1.0; off-diagonal should be ~0.3-0.7):")
print((video_embs @ video_embs.T).round(decimals=3))

# 5) Run text queries.
VIDEO_QUERIES = [
    "people cooking and preparing food in a kitchen",
    "cars and pedestrians at a city intersection",
    "a peaceful nature scene with tall trees",
    "water falling down rocks in a tropical setting",
]
results = []
for q in VIDEO_QUERIES:
    qemb = encode([{"text": q}])
    sims = (qemb @ video_embs.T)[0]
    ranking = sorted(enumerate(sims.tolist()), key=lambda x: -x[1])
    results.append({
        "query":   q,
        "ranking": [
            {"rank": r+1,
             "video": os.path.basename(video_paths[i]),
             "caption": video_captions[i],
             "score": round(s, 4)}
            for r, (i, s) in enumerate(ranking)
        ],
    })
print("\n" + json.dumps(results, indent=2))

In [ ]:
video_thumbs = [middle_frame(p) for p in video_paths]

for q in VIDEO_QUERIES:
    qemb   = encode([{"text": q}])
    sims   = (qemb @ video_embs.T)[0]
    ranked = sorted(enumerate(sims.tolist()), key=lambda x: -x[1])

    fig, axes = plt.subplots(
        1, len(VIDEO_CORPUS),
        figsize=(16, 3.2),
        gridspec_kw={"wspace": 0.05},
    )
    for col, (i, s) in enumerate(ranked):
        if video_thumbs[i] is not None:
            axes[col].imshow(video_thumbs[i])
        axes[col].set_title(
            f"#{col+1}  score={s:.3f}\n{video_captions[i][:35]}",
            fontsize=10,
        )
        axes[col].axis("off")

    fig.suptitle(f'Query: "{q}"', fontsize=13, y=0.97)
    plt.subplots_adjust(top=0.84, wspace=0.05, bottom=0.02)
    plt.show()
    print("-" * 80)

**What this shows.** Each text query lights up the matching clip with the highest score, and the wrong clips trail behind by a clear margin. The same `embedder.process(...)` call that handled images now handles videos — the model samples frames internally (`fps=1.0`, `max_frames=4` here so we fit on a T4) and folds them into one vector per clip.

**Middle-frame vs full-video, in one sentence:** the full video API is more accurate for temporal queries ("a person *stopping then* walking again") but costs roughly `max_frames` times more compute than encoding a single image (4× here, 16× at the default). For most retrieval pipelines, `fps=1.0` is a sweet spot — fine-grained enough to capture content changes, light enough to scale.

## 7c. Instruction-aware retrieval

Per the model card, Qwen3-VL-Embedding is *instruction-aware* and gains 1-5% on benchmarks when an instruction is provided in the input dict. The instruction is the model's way of being told *what kind* of similarity to compute.

The same query with different instructions can produce different rankings. We demonstrate by running the same plain query three ways: no instruction, "find food images", and "find images with red colors".

In [ ]:
# Same base query, three different instructions
BASE_QUERY = "a fruit"

variants = [
    {"text": BASE_QUERY},  # no instruction
    {"text": BASE_QUERY, "instruction": "Retrieve images that show edible food items."},
    {"text": BASE_QUERY, "instruction": "Retrieve images dominated by the color red."},
]

print("Three query variants (JSON):")
print(json.dumps(variants, indent=2))

print("\nTop-5 for each variant:")
all_rankings = []
for v in variants:
    qemb = encode([v])
    idxs, scores, _ = cosine_topk(qemb, image_embeddings, k=5)
    result = as_json_result(v.get("instruction", "<no instruction>"),
                            idxs, scores, captions=captions, urls=urls)
    print(json.dumps(result, indent=2))
    all_rankings.append((v.get("instruction", "no instruction"), idxs, scores))
    print("-" * 60)

In [ ]:

# One figure per instruction variant, with the instruction in the suptitle.
for instruction, idxs, scores in all_rankings:
    fig, axes = plt.subplots(
        1, 3,
        figsize=(12, 4.0),
        gridspec_kw={"wspace": 0.05},
    )
    for col, (i, s) in enumerate(zip(idxs[:3], scores[:3])):
        axes[col].imshow(images[i])
        axes[col].set_title(f"#{col+1}  score={s:.3f}\n{captions[i][:35]}", fontsize=10)
        axes[col].axis("off")

    title_text = (
        f'Base query: "{BASE_QUERY}"\nInstruction: {instruction}'
        if instruction != "no instruction"
        else f'Base query: "{BASE_QUERY}"\n(no instruction)'
    )
    fig.suptitle(title_text, fontsize=11, y=0.99)
    plt.subplots_adjust(top=0.74, wspace=0.05, bottom=0.02)
    plt.show()
    print("-" * 80)

With no instruction, the model returns generic fruit images. The "edible food" instruction pulls the assorted fruits image up. The "red colors" instruction tends to shuffle the red apple ahead of the orange. Same query, different rankings, driven entirely by the `instruction` field.

## 7d. Multilingual queries

Qwen3-VL-Embedding supports 30+ languages. The same image corpus can be searched in any of them. Here we run the *same semantic query* in five languages and compare the rankings.

In [ ]:
# Same semantic query in five languages
MULTILINGUAL = [
    ("english",  "a single red apple on a plain background"),
    ("hindi",    "एक सादे पृष्ठभूमि पर एक लाल सेब"),
    ("chinese",  "一个红苹果,简单背景"),
    ("spanish",  "una manzana roja sobre un fondo plano"),
    ("french",   "une pomme rouge sur un fond uni"),
]

print("Multilingual queries (JSON):")
print(json.dumps([{"lang": lang, "text": q} for lang, q in MULTILINGUAL],
                 indent=2, ensure_ascii=False))

print("\nTop-3 results per language:")
multi_results = []
for lang, q in MULTILINGUAL:
    qemb = encode([{"text": q}])
    idxs, scores, _ = cosine_topk(qemb, image_embeddings, k=3)
    multi_results.append({
        "lang":  lang,
        "query": q,
        "top_3": [
            {"caption": captions[i], "score": round(s, 4)}
            for i, s in zip(idxs, scores)
        ],
    })
print(json.dumps(multi_results, indent=2, ensure_ascii=False))

In [ ]:
fig, axes = plt.subplots(len(MULTILINGUAL), 3, figsize=(11, 3.3 * len(MULTILINGUAL)))
for row, (lang, q) in enumerate(MULTILINGUAL):
    qemb = encode([{"text": q}])
    idxs, scores, _ = cosine_topk(qemb, image_embeddings, k=3)
    for col in range(3):
        ax = axes[row, col]
        if col < len(idxs):
            ax.imshow(images[idxs[col]])
            ax.set_title(f"{scores[col]:.3f} | {captions[idxs[col]][:25]}", fontsize=8)
        ax.axis("off")
    axes[row, 0].set_ylabel(lang, fontsize=10, rotation=0, labelpad=40, ha="right")
plt.suptitle('"Red apple on plain background" in five languages', fontsize=11)
plt.tight_layout(); plt.show()

All five languages should rank the apple image (or fruits image) in the top three. The fact that you do not need to translate user queries before embedding makes this useful for global applications: index once in any language, search from any other.

### 7e. Pure text-to-text similarity

The same model also works as a strong text embedder. Multimodal models are sometimes weaker on text-only tasks than dedicated text encoders, but Qwen3-VL-Embedding is built on the Qwen3-VL foundation, which inherits the strong language understanding of the Qwen3 model family — so its text-only performance is competitive.

We build a tiny text corpus and run text queries against it — no images involved.

In [ ]:
TEXT_CORPUS = [
    "Modern neural networks for computer vision use convolutional layers.",
    "Transformers were originally introduced for machine translation tasks.",
    "Cosine similarity measures the angle between two high-dimensional vectors.",
    "The Renaissance was a cultural movement that began in Italy in the 14th century.",
    "Photosynthesis converts sunlight into chemical energy in green plants.",
    "Soccer is played on a rectangular field by two teams of eleven players.",
    "FAISS is an open-source library for efficient similarity search.",
    "Bach composed the Brandenburg Concertos in the early 18th century.",
]

print("Text corpus (8 paragraphs).")
text_embs = encode([{"text": t} for t in TEXT_CORPUS], batch_size=4)
print(f"Text corpus shape: {tuple(text_embs.shape)}")

TEXT_QUERIES = [
    "how do transformers relate to translation?",
    "what is a vector search library?",
    "the rules of football",
    "european art history",
]
for q in TEXT_QUERIES:
    qemb = encode([{"text": q}])
    sims = (qemb @ text_embs.T)[0]
    top = sims.argsort(descending=True)[:3].tolist()
    print(json.dumps({
        "query": q,
        "top_3": [
            {"score": round(sims[i].item(), 4), "text": TEXT_CORPUS[i]}
            for i in top
        ],
    }, indent=2))
    print()

## 8. The two-stage pipeline: Embedding then Reranker

This is the headline use case for the pair:

1. Use the Embedding model to find the top-K image candidates for a tricky query. Fast.
2. Re-score those K candidates with the Reranker. Slow per pair, but precise.
3. Compare the top-3 *before* and *after* re-ranking.

We pick a deliberately compositional query — the kind where bi-encoders are known to slip.

In [ ]:

# Stage 1: recall with the Embedding model
QUERY = "an outdoor scene completely empty with no people and no animals"
TOP_K = 10

q_emb = encode([{"text": QUERY}])
top_k_idxs, top_k_scores, _ = cosine_topk(q_emb, image_embeddings, k=TOP_K)

print(json.dumps({
    "stage":  1,
    "model":  "Qwen3-VL-Embedding-2B",
    "query":  QUERY,
    "top_k":  [
        {"rank": r+1, "index": i, "score": round(s, 4), "caption": captions[i]}
        for r, (i, s) in enumerate(zip(top_k_idxs, top_k_scores))
    ],
}, indent=2))

# 2 x 5 grid with word-wrapped captions so nothing gets cut mid-word
rows, cols = 2, 5
fig, axes = plt.subplots(
    rows, cols,
    figsize=(16, 7.5),
    gridspec_kw={"wspace": 0.05, "hspace": 0.45},
)
for ax, i, s in zip(axes.flatten(), top_k_idxs, top_k_scores):
    ax.imshow(images[i])
    wrapped = textwrap.fill(captions[i], width=22, max_lines=2, placeholder="…")
    ax.set_title(f"{s:.3f}\n{wrapped}", fontsize=8)
    ax.axis("off")

plt.suptitle(f"Stage 1 (Embedding) recall: top-{TOP_K}", fontsize=12, y=0.99)
plt.subplots_adjust(top=0.88, wspace=0.05, hspace=0.45)
plt.show()

In [ ]:
# Stage 2: rerank those K candidates
documents = [{"image": urls[i]} for i in top_k_idxs]

rr_inputs = {
    "instruction": "Retrieve images that match the user's description.",
    "query":       {"text": QUERY},
    "documents":   documents,
    "fps":         1.0,
    "max_frames":  64,
}
print("Reranker input (JSON, documents truncated):")
print(json.dumps({**rr_inputs, "documents": rr_inputs["documents"][:3] + ["..."]}, indent=2))

print("\nScoring with reranker (per-pair, slower than embedding) ...")
t0 = time.time()
rerank_scores = reranker.process(rr_inputs)
print(f"Reranked {len(documents)} candidates in {time.time() - t0:.1f} s")

reranked = sorted(zip(top_k_idxs, top_k_scores, rerank_scores), key=lambda x: -x[2])
print("\nFinal ranking (JSON):")
print(json.dumps({
    "stage": 2,
    "model": "Qwen3-VL-Reranker-2B",
    "query": QUERY,
    "top_k": [
        {
            "rank":           r+1,
            "index":          i,
            "embed_score":    round(es, 4),
            "rerank_score":   round(rs, 4),
            "caption":        captions[i],
        }
        for r, (i, es, rs) in enumerate(reranked[:5])
    ],
}, indent=2))

In [ ]:
# Side-by-side visualisation: top-3 before and after rerank
top3_embed  = top_k_idxs[:3]
top3_rerank = [i for i, _, _ in reranked[:3]]

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for col, i in enumerate(top3_embed):
    axes[0, col].imshow(images[i]); axes[0, col].axis("off")
    axes[0, col].set_title(f"embed only #{col+1}\n{captions[i][:30]}", fontsize=9)
for col, i in enumerate(top3_rerank):
    axes[1, col].imshow(images[i]); axes[1, col].axis("off")
    axes[1, col].set_title(f"after rerank #{col+1}\n{captions[i][:30]}",
                           fontsize=9, color="darkgreen")
plt.suptitle(f'Query: "{QUERY}"', fontsize=12)
plt.tight_layout(); plt.show()

## 9. Try your own query

Edit the cell below and run it. The query goes through the full two-stage pipeline against the image corpus we built. You can swap in your own image URLs in `CORPUS` at the top of the notebook to search a different corpus.

In [ ]:
# === Widgets ===
query_box = widgets.Text(
    value="a calm peaceful outdoor scene",
    placeholder="Type your query here",
    description="Query:",
    layout=widgets.Layout(width="700px"),
)
recall_slider = widgets.IntSlider(
    value=10, min=3, max=20, step=1,
    description="Top-K recall:",
    layout=widgets.Layout(width="450px"),
)
final_slider = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description="Top-K final:",
    layout=widgets.Layout(width="450px"),
)
run_button = widgets.Button(description="Run pipeline", button_style="primary")
output_area = widgets.Output()

# === Callback ===
def run_pipeline(_):
    with output_area:
        clear_output(wait=True)
        MY_QUERY     = query_box.value
        TOP_K_RECALL = recall_slider.value
        TOP_K_FINAL  = final_slider.value

        # Stage 1: recall
        q_emb           = encode([{"text": MY_QUERY}])
        cand_idxs, _, _ = cosine_topk(q_emb, image_embeddings, k=TOP_K_RECALL)

        # Stage 2: rerank
        rr_inputs = {
            "instruction": "Retrieve images that match the user's description.",
            "query":       {"text": MY_QUERY},
            "documents":   [{"image": urls[i]} for i in cand_idxs],
            "fps":         1.0,
            "max_frames":  64,
        }
        rr_scores = reranker.process(rr_inputs)
        final     = sorted(zip(cand_idxs, rr_scores), key=lambda x: -x[1])[:TOP_K_FINAL]

        # JSON output
        print(json.dumps({
            "query":  MY_QUERY,
            "top_k":  [
                {"rank": r+1, "rerank_score": round(s, 4), "caption": captions[i]}
                for r, (i, s) in enumerate(final)
            ],
        }, indent=2))

        # Visualization
        fig, axes = plt.subplots(1, TOP_K_FINAL, figsize=(4 * TOP_K_FINAL, 4))
        axes = [axes] if TOP_K_FINAL == 1 else axes
        for ax, (i, s) in zip(axes, final):
            ax.imshow(images[i])
            ax.set_title(f"rerank={s:.3f}\n{captions[i][:35]}", fontsize=10)
            ax.axis("off")
        plt.suptitle(f'Your query: "{MY_QUERY}"', fontsize=12)
        plt.tight_layout()
        plt.show()

run_button.on_click(run_pipeline)

# === Display the form ===
display(widgets.VBox([
    widgets.HTML("<b>Try your own query</b><br>Enter text, adjust K values, then click Run."),
    query_box,
    recall_slider,
    final_slider,
    run_button,
    output_area,
]))

## Wrap-up

You have walked the full Qwen3-VL retrieval suite end to end:

- The **Embedding** model gives every item (text, image, screenshot, video, or any mix) a 2048-dim L2-normalised vector. Indexing is done once; queries are dot products.
- The **Reranker** takes `(query, document)` pairs and returns precise relevance scores in `[0, 1]`, at higher cost per pair.
- The two are designed to be **used together**: embedding for recall, reranker for precision.

Capabilities demonstrated:

1. Text-to-image retrieval
2. Image-to-image retrieval
3. Logo / screenshot / document retrieval
4. Mixed-modal queries (image + text in one vector)
5. Short-video clip retrieval (middle-frame trick)
6. Full video API (`{"video": path, "fps": ...}`)
7. Instruction-aware retrieval
8. Multilingual queries
9. Pure text-to-text similarity
10. Two-stage Embedding → Reranker pipeline

### Where to go from here

- Swap the OpenCV-samples corpus for your own (a screenshot folder, a product catalogue, a video archive).
- For larger corpora, store the image embeddings in a vector store (FAISS, Milvus, Qdrant) and only run the reranker on the top-K candidates returned by the vector store.
- The 8B variants of both models push the numbers higher (MMEB-V2 of 77.8 for Embedding-8B), at the cost of roughly double the VRAM. Use them when 2B accuracy is the bottleneck.

### Read more

- Paper: [Qwen3-VL-Embedding and Qwen3-VL-Reranker (arXiv:2601.04720)](https://arxiv.org/abs/2601.04720)
- Embedding model card: [Qwen/Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B)
- Reranker model card: [Qwen/Qwen3-VL-Reranker-2B](https://huggingface.co/Qwen/Qwen3-VL-Reranker-2B)
- Code: [QwenLM/Qwen3-VL-Embedding](https://github.com/QwenLM/Qwen3-VL-Embedding)